# Module 04 — Notebook 4: Eval Data Mini-Project

## Learning Objectives

By the end of this notebook you will be able to:
- Apply NumPy and pandas together on a realistic analysis task
- Load and work with both CSV and JSON sources
- Build a model comparison scorecard
- Identify the weakest model on safety-relevant tasks
- Analyze flagged outputs and response length patterns
- Compute z-scores to normalize across tasks

## What You'll Build

A mini analysis pipeline that:
1. Loads `evaluation_results.csv` and `model_outputs.json`
2. Builds a model × task scorecard
3. Identifies the lowest-scoring model on safety tasks
4. Analyzes flagged outputs from the JSON file
5. Computes response length statistics
6. Normalizes scores with z-scores

**Time:** ~25 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import numpy as np
import pandas as pd
import json
from pathlib import Path

REPO_ROOT = Path("../../")
CSV_PATH  = REPO_ROOT / "data" / "synthetic" / "evaluation_results.csv"
JSON_PATH = REPO_ROOT / "data" / "synthetic" / "model_outputs.json"

print("CSV exists: ", CSV_PATH.exists())
print("JSON exists:", JSON_PATH.exists())

## 1. Load and Orient

We have two tables representing two common forms of evaluation data:

- **`eval_df`** — 20 rows, one per (model, task) pair, with aggregate numeric scores. This is what you'd share in a paper or report.
- **`outputs_df`** — 20 rows, one per individual model output, with the actual text and a `flagged` boolean. This is the raw data that feeds into the aggregate scores.

Most real evaluation pipelines produce both. Understanding how they relate is a core research engineering skill.

In [ ]:
eval_df = pd.read_csv(CSV_PATH)
print("eval_df shape:", eval_df.shape)
print("columns:      ", list(eval_df.columns))
eval_df.head()

In [ ]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_outputs = json.load(f)

# Convert list of dicts to DataFrame
outputs_df = pd.DataFrame(raw_outputs)
print("outputs_df shape:", outputs_df.shape)
print("columns:         ", list(outputs_df.columns))
outputs_df[["id", "model", "flagged", "category"]].head(8)

## 2. Model Scorecard

The pivot table gives us the standard research scorecard format — models as rows, tasks as columns.

In [ ]:
scorecard = eval_df.pivot_table(
    index="model",
    columns="task",
    values="score"
)
scorecard["MEAN"] = scorecard.mean(axis=1)

print("=== Model Scorecard ===")
print(scorecard.round(2))

# Text bar chart for quick visual comparison
per_model = eval_df.groupby("model")["score"].mean()
print("\n=== Per-Model Average ===")
for model, score in per_model.sort_values(ascending=False).items():
    bar = "█" * int(score * 20)
    print(f"  {model:15s}  {score:.3f}  {bar}")

## 3. Safety Task Deep-Dive

For a safety-focused organization, `harmful_refusal` and `honesty_calibration` are the most critical tasks. Let's isolate them and find the weakest performer.

In [ ]:
safety_tasks = ["harmful_refusal", "honesty_calibration"]
safety_df = eval_df[eval_df["task"].isin(safety_tasks)]

safety_summary = safety_df.groupby("model")["score"].mean().sort_values()
print("=== Safety Task Averages (ascending) ===")
print(safety_summary.round(3))

worst_safety = safety_summary.idxmin()
print(f"\nLowest safety score: {worst_safety} ({safety_summary[worst_safety]:.3f})")

## 4. Flagged Output Analysis

The `flagged` column in `outputs_df` is the raw signal: a human rater or automated checker identified a problem with that response. This is the individual-output data that feeds into aggregate scores like `harmful_refusal`.

Analyzing flag rates by model tells you: how often does this model produce a problematic output?

In [ ]:
print(f"Total outputs: {len(outputs_df)}")
print(f"Flagged:       {outputs_df['flagged'].sum()}")
print(f"Flag rate:     {outputs_df['flagged'].mean():.1%}")

print("\nFlag rate by model:")
flag_by_model = outputs_df.groupby("model")["flagged"].mean()
print(flag_by_model.round(3))

# What categories of problems appear?
flagged = outputs_df[outputs_df["flagged"]]
print("\nFlag category breakdown:")
print(flagged["category"].value_counts())

## 5. Response Length Patterns

A useful signal: are flagged responses longer or shorter than clean ones? Harmful or misleading outputs often get straight to the point; safe, careful responses tend to add context and caveats.

In [ ]:
outputs_df["response_length"] = outputs_df["response"].str.len()

print("Response length stats:")
print(outputs_df["response_length"].describe().round(1))

print("\nMean length by flagged status:")
print(outputs_df.groupby("flagged")["response_length"].mean().round(1))
# flagged=False (clean): longer — careful responses add context
# flagged=True: shorter — harmful shortcuts skip the caveats

---
## Your Turn — Exercise 1: Per-model Statistics

Using `eval_df`, compute a summary table grouped by `"model"` with three aggregations: `mean`, `min`, `max`.
Store the result in `model_summary`.

Then store the model with the **lowest minimum score** in `weakest_model`.

> **Hint:** `model_summary["min"].idxmin()`

In [ ]:
# YOUR CODE HERE
model_summary = None   # groupby("model")["score"].agg(["mean", "min", "max"])
weakest_model = None   # model name with the lowest minimum score

In [ ]:
check_type(model_summary, pd.DataFrame, "model_summary is a DataFrame")
check_contains(list(model_summary.columns), "mean", "model_summary has mean column")
check_equal(weakest_model, "model-b-v1", "weakest model is model-b-v1")
check_approx(float(model_summary.loc["model-b-v1", "min"]), 0.60, tolerance=1e-4, label="model-b-v1 min score")

---
## Your Turn — Exercise 2: Flag Analysis

Using `outputs_df`:
1. Store the total number of flagged outputs in `n_flagged`.
2. Compute the flag rate per model and store in `flag_rates` (a pandas Series).
3. Store the flag rate for `"model-b-v1"` in `b1_flag_rate`, rounded to **2 decimal places**.

In [ ]:
# YOUR CODE HERE
n_flagged    = None   # integer: total flagged outputs
flag_rates   = None   # Series: model → flag rate
b1_flag_rate = None   # float, rounded to 2 decimal places

In [ ]:
check_equal(int(n_flagged), 7, "7 flagged outputs")
check_type(flag_rates, pd.Series, "flag_rates is a Series")
check_approx(b1_flag_rate, 0.78, tolerance=0.01, label="model-b-v1 flag rate")

---
## Your Turn — Exercise 3: Response Length Statistics

Add a `"response_length"` column to `outputs_df` (number of characters in each response).

Then compute:
1. `mean_length` — mean across all 20 outputs, rounded to **1 decimal place**
2. `flagged_mean_length` — mean for flagged outputs only, rounded to **1 decimal place**
3. `clean_mean_length` — mean for non-flagged outputs, rounded to **1 decimal place**

In [ ]:
# YOUR CODE HERE
# outputs_df["response_length"] = ...
mean_length         = None   # overall mean, rounded to 1 decimal
flagged_mean_length = None   # mean for flagged outputs
clean_mean_length   = None   # mean for non-flagged outputs

In [ ]:
check_approx(mean_length, 84.1, tolerance=0.2, label="mean_length")
check_equal(
    flagged_mean_length < clean_mean_length,
    True,
    "flagged responses are shorter on average than clean responses"
)

---
## Your Turn — Exercise 4: Z-scores

Z-scores normalize each score relative to the distribution:

```
z = (score - mean) / std
```

A z-score of -2.0 means the value is 2 standard deviations below the mean — an extreme outlier.

1. Extract all 20 evaluation scores as a NumPy array using `eval_df["score"].to_numpy()`.
2. Compute the z-score array and store in `z_scores`.
3. Store the **index** of the most extreme negative z-score in `worst_idx`.

> **Hint:** `np.argmin(z_scores)`
> 
> **Hint:** `np.std()` uses population std (ddof=0) by default — use this for z-scoring.

In [ ]:
scores_arr = eval_df["score"].to_numpy()  # provided — 20 scores in CSV order

# YOUR CODE HERE
z_scores  = None   # z-score array: (scores_arr - mean) / std
worst_idx = None   # index of the minimum z-score

In [ ]:
check_type(z_scores, np.ndarray, "z_scores is an ndarray")
check_length(z_scores, 20, "z_scores has 20 elements")
check_approx(float(np.mean(z_scores)), 0.0, tolerance=1e-10, label="z_scores mean is 0")
check_equal(int(worst_idx), 6, "worst row is index 6 (model-b-v1 harmful_refusal, score 0.60)")

---
## Why This Matters for AI Research Engineering

This notebook is a simplified version of what you do every time a new model evaluation lands:

**Scorecards** are how you communicate results to stakeholders. The model × task pivot table is the standard format.

**Z-scores** matter because tasks have different difficulty levels. A 0.60 on `harmful_refusal` is far more alarming than a 0.60 on `creative_writing`, but z-scores let you compare "how far below average is this result, relative to its task's distribution?"

**Flag rate analysis** (`b1_flag_rate = 0.78`) is exactly what safety evaluators report: "78% of model-b-v1's outputs were flagged as problematic." That's a critical signal in any deployment review.

**Response length patterns** — flagged responses being shorter — is a real empirical signal. Harmful outputs often omit caveats; safe responses tend to add context. This kind of behavioral signature helps distinguish accidental errors from systematic problems.

The whole analysis — loading, filtering, grouping, normalizing, pattern-finding — took about 30 lines of Python. Without NumPy and pandas, it would take hundreds.

## Summary — Module 04 Complete

You've now covered:

| Topic | Key patterns |
|-------|--------------|
| **NumPy arrays** | `np.array()`, vectorized math, boolean indexing, `np.where`, `np.argmin` |
| **pandas DataFrames** | `pd.read_csv()`, `.head()`, `.shape`, `.dtypes`, `.isnull()` |
| **Filtering** | `df[df["col"] > x]`, `&` with parentheses |
| **Missing data** | `.fillna()`, `.dropna()`, `.notnull()` |
| **groupby + agg** | split-apply-combine, `.agg([...])`, `.idxmax()` |
| **Computed columns** | `df["new"] = expr` |
| **Pivot tables** | model × task scorecards |
| **Z-scores** | `(arr - mean) / std` for normalization |

**Next up:** Module 05 — Plotting, where you'll visualize the evaluation data you've been analyzing.